# 📥 FHIR Raw Data Ingestion

## Overview
This notebook ingests FHIR (Fast Healthcare Interoperability Resources) data from a FHIR server and stores it in the raw data layer of our medallion architecture.

## What It Does
1. **Loads configuration** from `config/resources.json`
2. **Fetches FHIR resources** (Patient, Encounter, Observation, Condition) from the API
3. **Handles pagination** automatically with retry logic
4. **Stores raw responses** with full traceability metadata
5. **Maintains referential integrity** by fetching resources in dependency order

## Configuration
- **Base URL**: FHIR server endpoint (e.g., https://hapi.fhir.org/baseR4)
- **Resources**: List of FHIR resource types to fetch
- **Page Count**: Number of records per API page
- **Lookback Days**: How many days back to fetch updated records

## Output Structure
```
data/raw/
  └── YYYY-MM-DD/           # Extraction date
      └── {ResourceType}/    # e.g., Patient, Observation
          ├── page_0000.json
          ├── page_0001.json
          └── ...
```

## Features
✅ **Resilient**: HTTP retry logic with exponential backoff  
✅ **Traceable**: Full metadata captured for each API call  
✅ **Modular**: Reusable utilities in separate module  
✅ **Production-ready**: Proper error handling and logging  

## Usage
Simply run all cells in order. The notebook will:
1. Load configuration
2. Set up HTTP session with retry logic
3. Fetch and store all configured resources
4. Display ingestion summary

---
*Part of the FHIR Medallion Lakehouse pipeline*

In [0]:
# Standard library imports
import json
import logging
from datetime import datetime, timedelta, UTC
from pathlib import Path
from typing import Dict, List, Optional, Any

# Third-party imports
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
display(logger)

In [0]:
# Configuration
# In production, load from Unity Catalog or external config service
config = {
    "base_url": "https://hapi.fhir.org/baseR4",
    "resources": ["Patient", "Encounter", "Observation", "Condition"],
    "page_count": 20,
    "lookback_days": 3
}
logger.info("Configuration loaded successfully")

# Extract configuration values
BASE_URL: str = config["base_url"]
RESOURCES: List[str] = config["resources"]
PAGE_COUNT: int = config["page_count"]
LOOKBACK_DAYS: int = config["lookback_days"]

# Get the project root directory for data storage
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = str(Path(notebook_path).parent.parent)

# Set raw data root - using file: scheme for local workspace storage
RAW_ROOT: str = f"file:/Workspace{project_root}/data/raw"

logger.info(f"Project root: {project_root}")
logger.info(f"Base URL: {BASE_URL}")
logger.info(f"Resources: {RESOURCES}")
logger.info(f"Page count: {PAGE_COUNT}")
logger.info(f"Lookback days: {LOOKBACK_DAYS}")
logger.info(f"Raw data root: {RAW_ROOT}")

In [0]:
def create_http_session(max_retries: int = 3, backoff_factor: float = 0.5) -> requests.Session:
    """
    Create a requests session with retry logic for resilient API calls.
    
    Args:
        max_retries: Maximum number of retry attempts
        backoff_factor: Backoff factor for exponential backoff
    
    Returns:
        Configured requests.Session object
    """
    session = requests.Session()
    
    retry_strategy = Retry(
        total=max_retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    # Set default headers
    session.headers.update({
        "Accept": "application/fhir+json",
        "User-Agent": "FHIR-Medallion-Lakehouse/1.0"
    })
    
    return session

# Create global session for reuse
http_session = create_http_session()
logger.info("HTTP session configured with retry logic")

In [0]:
class FHIRIngestionError(Exception):
    """Custom exception for FHIR ingestion errors."""
    pass


def fetch_resource(
    resource_name: str,
    since_date: str,
    base_url: str,
    page_count: int,
    session: requests.Session
) -> List[Dict[str, Any]]:
    """
    Fetch all pages for one FHIR resource updated since the given date.
    
    Args:
        resource_name: FHIR resource type (e.g., 'Patient', 'Observation')
        since_date: ISO date string for filtering (YYYY-MM-DD)
        base_url: Base URL of the FHIR server
        page_count: Number of records per page
        session: Configured requests.Session object
    
    Returns:
        List of all resource entries across all pages
    
    Raises:
        FHIRIngestionError: If API request fails
    """
    url = f"{base_url}/{resource_name}?_lastUpdated=ge{since_date}&_count={page_count}"
    all_entries = []
    page_num = 0
    call_timestamp = datetime.now(UTC).isoformat()
    
    logger.info(f"Starting fetch for {resource_name} since {since_date}")
    
    try:
        while url:
            logger.debug(f"Fetching page {page_num} from: {url}")
            
            response = session.get(url, timeout=30)
            response.raise_for_status()
            bundle = response.json()
            
            entries = bundle.get("entry", [])
            all_entries.extend(entries)
            
            # Save raw page for full traceability
            save_raw_page(
                resource_name=resource_name,
                page_num=page_num,
                bundle_json=bundle,
                call_timestamp=call_timestamp,
                api_url=url
            )
            
            # Follow the "next" link if present
            next_link = next(
                (link["url"] for link in bundle.get("link", []) 
                 if link.get("relation") == "next"),
                None
            )
            
            url = next_link
            page_num += 1
            
            logger.debug(f"Page {page_num - 1}: fetched {len(entries)} entries")
    
    except requests.exceptions.RequestException as e:
        error_msg = f"Failed to fetch {resource_name}: {str(e)}"
        logger.error(error_msg)
        raise FHIRIngestionError(error_msg) from e
    except json.JSONDecodeError as e:
        error_msg = f"Invalid JSON response for {resource_name}: {str(e)}"
        logger.error(error_msg)
        raise FHIRIngestionError(error_msg) from e
    
    logger.info(f"Completed fetch for {resource_name}: {len(all_entries)} total entries across {page_num} pages")
    return all_entries


def save_raw_page(
    resource_name: str,
    page_num: int,
    bundle_json: Dict[str, Any],
    call_timestamp: str,
    api_url: str
) -> str:
    """
    Persist raw API response with metadata.
    
    Args:
        resource_name: FHIR resource type
        page_num: Page number (0-indexed)
        bundle_json: Raw FHIR bundle response
        call_timestamp: ISO timestamp of the API call
        api_url: Full API URL that was called
    
    Returns:
        Path where the file was saved
    
    Raises:
        FHIRIngestionError: If file write fails
    """
    try:
        extraction_date = datetime.now(UTC).strftime("%Y-%m-%d")
        folder = f"{RAW_ROOT}/{extraction_date}/{resource_name}"
        
        # Create directory if it doesn't exist
        dbutils.fs.mkdirs(folder)
        
        # Build payload with metadata
        payload = {
            "resource_type": resource_name,
            "extraction_timestamp": call_timestamp,
            "extraction_date": extraction_date,
            "api_url": api_url,
            "page_number": page_num,
            "entry_count": len(bundle_json.get("entry", [])),
            "response": bundle_json
        }
        
        # Write to file
        out_path = f"{folder}/page_{page_num:04d}.json"
        dbutils.fs.put(out_path, json.dumps(payload, indent=2), overwrite=True)
        
        logger.debug(f"Saved page {page_num} to: {out_path}")
        return out_path
        
    except Exception as e:
        error_msg = f"Failed to save page {page_num} for {resource_name}: {str(e)}"
        logger.error(error_msg)
        raise FHIRIngestionError(error_msg) from e

In [0]:
def run_ingestion(
    resources: List[str],
    lookback_days: int,
    base_url: str,
    page_count: int,
    session: requests.Session
) -> Dict[str, Any]:
    """
    Execute the full ingestion pipeline for all resources.
    
    Args:
        resources: List of FHIR resource types to fetch
        lookback_days: Number of days to look back for updated records
        base_url: Base URL of the FHIR server
        page_count: Number of records per page
        session: Configured requests.Session object
    
    Returns:
        Dictionary with ingestion statistics
    """
    since_date = (datetime.now(UTC) - timedelta(days=lookback_days)).strftime("%Y-%m-%d")
    
    logger.info("="*60)
    logger.info("FHIR RAW DATA INGESTION STARTED")
    logger.info("="*60)
    logger.info(f"Lookback period: {lookback_days} days (since {since_date})")
    logger.info(f"Resources to fetch: {', '.join(resources)}")
    logger.info("="*60)
    
    results = {
        "start_time": datetime.now(UTC).isoformat(),
        "since_date": since_date,
        "resources": {},
        "total_records": 0,
        "errors": []
    }
    
    # Fetch each resource in order (maintains referential integrity)
    for resource in resources:
        try:
            logger.info(f"\n[{resource}] Starting ingestion...")
            
            entries = fetch_resource(
                resource_name=resource,
                since_date=since_date,
                base_url=base_url,
                page_count=page_count,
                session=session
            )
            
            record_count = len(entries)
            results["resources"][resource] = {
                "status": "success",
                "record_count": record_count
            }
            results["total_records"] += record_count
            
            logger.info(f"[{resource}] ✓ Completed: {record_count} records")
            
        except FHIRIngestionError as e:
            error_msg = f"Failed to fetch {resource}: {str(e)}"
            logger.error(f"[{resource}] ✗ {error_msg}")
            
            results["resources"][resource] = {
                "status": "failed",
                "error": str(e)
            }
            results["errors"].append(error_msg)
            
            # Continue with next resource instead of failing completely
            continue
    
    results["end_time"] = datetime.now(UTC).isoformat()
    
    logger.info("\n" + "="*60)
    logger.info("FHIR RAW DATA INGESTION COMPLETED")
    logger.info("="*60)
    logger.info(f"Total records ingested: {results['total_records']}")
    logger.info(f"Successful: {sum(1 for r in results['resources'].values() if r['status'] == 'success')} / {len(resources)}")
    
    if results["errors"]:
        logger.warning(f"Errors encountered: {len(results['errors'])}")
    
    logger.info("="*60)
    
    return results


# Execute ingestion
ingestion_results = run_ingestion(
    resources=RESOURCES,
    lookback_days=LOOKBACK_DAYS,
    base_url=BASE_URL,
    page_count=PAGE_COUNT,
    session=http_session
)

# Display summary
display(ingestion_results)